# {{catalog_name}} — Catalog QA Diagnostics

Quality-assurance diagnostics for the DES Y6 WaZP cluster catalog.

**Sample:** full catalog (or subset defined by the catalog filters).

**References**
- [Benoist et al. 2025, A&A](https://doi.org/10.1051/0004-6361/202555607) — DES Y6 WaZP catalog
- [Aguena et al. 2021, MNRAS 502, 4435](https://doi.org/10.1093/mnras/stab264) — WaZP method
- [Castignani & Benoist 2016, A&A 595, A111](https://doi.org/10.1051/0004-6361/201528009) — membership probability $P_{\mathrm{mem}}$
- [LIneA WaZP data products](https://data.linea.org.br/en/sci_products/wazp.html)

**Cluster properties shown:** photometric redshift $z_{\mathrm{phot}}$, richness $N_{\mathrm{gals}}$, signal-to-noise ratio $\mathrm{SNR}$, cluster radius $R_{\mathrm{amin}}$, sky coordinates, and SZE cross-match flags (ACT, SPT).

In [ ]:
# canvas-variables
import json
table_metadata = json.loads("""{{table_metadata}}""")


In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dblinea import MyDB

warnings.filterwarnings("ignore")

# Plotting defaults
plt.rcParams["figure.dpi"] = 100
FIG_WIDTH = 12.0

# LaTeX labels — consistent with Benoist et al. (2025) and Aguena et al. (2021)
LBL = {
    "zphot": r"$z_{\mathrm{phot}}$",
    "ngals": r"$N_{\mathrm{gals}}$",
    "snr": r"$\mathrm{SNR}$",
    "radius_mpc": r"$R_{\mathrm{amin}}$ (Mpc)",
    "counts": "Count",
    "density": r"$\Sigma$ (arcmin$^{-2}$)",
    "ra": "RA (deg)",
    "dec": "Dec (deg)",
    "cumulative": r"$N(>z)$",
}

# --- Column discovery from table metadata ---
schema = table_metadata["schema"]
table_name = table_metadata["table"]
full_table_name = f"{schema}.{table_name}"
columns = {c["name"]: c for c in table_metadata["columns"]}
ucd_to_col = {c["ucd"]: c["name"] for c in table_metadata["columns"] if c.get("ucd")}


def col_by_ucd(ucd, fallback=None):
    return ucd_to_col.get(ucd, fallback)


def col_by_name(name):
    return name if name in columns else None


id_col = col_by_ucd("meta.id;meta.main", "id")
ra_col = col_by_ucd("pos.eq.ra;meta.main", "ra")
dec_col = col_by_ucd("pos.eq.dec;meta.main", "dec")
z_col = col_by_name("zphot") or col_by_ucd("src.redshift.phot", "zphot")
ngals_col = col_by_name("ngals") or col_by_ucd("meta.number", "ngals")
snr_col = col_by_name("snr") or col_by_ucd("stat.snr", "snr")
radius_col = col_by_name("radius_mpc") or col_by_ucd("phys.size.radius", "radius_mpc")
in_cosmo_col = col_by_name("in_cosmo") or "in_cosmo"
sze_act_col = col_by_name("sze_act_id") or "sze_act_id"
sze_spt_col = col_by_name("sze_spt_id") or "sze_spt_id"

print(f"Table: {full_table_name}")
print(f"Columns: id={id_col}, ra={ra_col}, dec={dec_col}, z={z_col}, "
      f"ngals={ngals_col}, snr={snr_col}, radius={radius_col}")

In [ ]:
db = MyDB(username=table_metadata["owner"])

agg_cols = [c for c in [z_col, ngals_col, snr_col, radius_col] if c]
agg_select = ", ".join(
    [f"MIN({c}) AS min_{c}, MAX({c}) AS max_{c}, AVG({c}) AS avg_{c}, COUNT({c}) AS n_{c}" for c in agg_cols]
)

query = f"SELECT COUNT(*) AS n_total, {agg_select} FROM {full_table_name}"
stats = pd.DataFrame(db.fetchall_dict(query))
row = stats.iloc[0]

print("Catalog summary statistics")
print("=" * 62)
print(f"  Total clusters:               {int(row['n_total']):,}")
for c in agg_cols:
    label = LBL.get(c, c)
    print(f"  {label:30s}  min={row[f'min_{c}']:>10.4f}  "
          f"max={row[f'max_{c}']:>10.4f}  mean={row[f'avg_{c}']:>10.4f}  "
          f"non-null={int(row[f'n_{c}']):,}")

## Redshift distribution

Photometric redshift distribution for different richness thresholds $N_{\mathrm{gals}} \geq N_{\mathrm{th}}$. The bin width $\Delta z = 0.05$ follows the DES Y6 photometric redshift precision (Benoist et al. 2025).

$$0 \leq z_{\mathrm{phot}} < 1.2, \quad \Delta z = 0.05$$

In [ ]:
ngals_th = (20, 25, 40)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ngals_th)))

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))

for n, color in zip(ngals_th, colors):
    query = f"""
    SELECT width_bucket({z_col}, 0, 1.2, 24) AS bin, COUNT(*) AS n
    FROM {full_table_name}
    WHERE {ngals_col} >= {n} AND {z_col} IS NOT NULL
    GROUP BY bin ORDER BY bin
    """
    df = pd.DataFrame(db.fetchall_dict(query))
    if df.empty:
        continue
    bin_edges = np.arange(0, 1.25, 0.05)
    counts = np.zeros(len(bin_edges) - 1)
    for _, row in df.iterrows():
        idx = int(row["bin"]) - 1
        if 0 <= idx < len(counts):
            counts[idx] = row["n"]
    ax.step(
        bin_edges[:-1], counts, where="post", lw=2, color=color,
        label=rf"$N_{{\mathrm{{gals}}}} \geq {n}$  ($n = {int(counts.sum()):,}$)",
    )

ax.set_xlabel(LBL["zphot"])
ax.set_ylabel(LBL["counts"])
ax.set_xlim(0, 1.2)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Richness distribution

Richness $N_{\mathrm{gals}}$ distribution for different signal-to-noise thresholds $\mathrm{SNR} \geq S_{\mathrm{th}}$. The richness $N_{\mathrm{gals}}$ is the number of galaxy members assigned by the WaZP algorithm in the $z$ band (Aguena et al. 2021; Benoist et al. 2025). Logarithmic binning is used to capture the power-law behaviour at the high-richness tail.

$$\Delta \log_{10} N_{\mathrm{gals}} \quad \text{(80 log-spaced bins)}$$

In [ ]:
snr_th = (3, 4, 5)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(snr_th)))

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))

query = f"""
SELECT MIN({ngals_col}) AS min_ngals, MAX({ngals_col}) AS max_ngals
FROM {full_table_name}
WHERE {ngals_col} IS NOT NULL
"""
range_df = pd.DataFrame(db.fetchall_dict(query))
min_ngals = max(range_df["min_ngals"].iloc[0], 1)
max_ngals = range_df["max_ngals"].iloc[0]
log_bins = np.logspace(np.log10(min_ngals), np.log10(max_ngals), 80)

for s, color in zip(snr_th, colors):
    query = f"""
    SELECT {ngals_col} FROM {full_table_name}
    WHERE {snr_col} >= {s} AND {ngals_col} IS NOT NULL
    """
    df = pd.DataFrame(db.fetchall_dict(query))
    if df.empty:
        continue
    vals = df[ngals_col].astype(float)
    ax.hist(
        vals, bins=log_bins, histtype="step", lw=2, color=color,
        label=rf"$\mathrm{{SNR}} \geq {s}$  ($n = {len(vals):,}$)",
    )

ax.set_xscale("log")
ax.set_xlabel(LBL["ngals"])
ax.set_ylabel(LBL["counts"])
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Redshift–richness plane

Cluster distribution in the $z_{\mathrm{phot}}$–$N_{\mathrm{gals}}$ plane, color-coded by signal-to-noise ratio $\mathrm{SNR}$. A random subsample of up to 5 000 clusters is shown to avoid overcrowding (Aguena et al. 2021). The richness dynamic range and redshift coverage reflect the DES Y6 selection function (Benoist et al. 2025).

In [ ]:
query = f"""
SELECT {z_col}, {ngals_col}, {snr_col}
FROM {full_table_name}
WHERE {z_col} IS NOT NULL AND {ngals_col} IS NOT NULL AND {snr_col} IS NOT NULL
ORDER BY RANDOM()
LIMIT 5000
"""
df = pd.DataFrame(db.fetchall_dict(query))

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))
order = np.argsort(df[snr_col].astype(float))
z = df[z_col].astype(float).iloc[order].values
ng = df[ngals_col].astype(float).iloc[order].values
snr = df[snr_col].astype(float).iloc[order].values

# Log-normalised color scale for SNR
c = (np.log10(snr) - np.log10(snr.min())) / (np.log10(snr.max()) - np.log10(snr.min()))
sc = ax.scatter(z, ng, c=c, s=8, marker=".", cmap="plasma", vmin=0, vmax=1)
ax.set_yscale("log")
ax.set_xlabel(LBL["zphot"])
ax.set_ylabel(LBL["ngals"])
cb = plt.colorbar(sc, ax=ax, label=r"$\log_{10} \mathrm{SNR}$ (normalised)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sky distribution

Spatial density map in HEALPix projection ($N_{\mathrm{side}} = 32$, resolution $\approx 1.8^\circ$). The color scale shows cluster surface density $\Sigma$ (arcmin$^{-2}$). This map reveals the DES Y6 footprint geometry and any large-scale projection systematics (Benoist et al. 2025).

In [ ]:
import healpy as hp

nside = 32
query = f"SELECT {ra_col}, {dec_col} FROM {full_table_name} WHERE {ra_col} IS NOT NULL AND {dec_col} IS NOT NULL"
df = pd.DataFrame(db.fetchall_dict(query))
ra = np.asarray(df[ra_col], dtype=float)
dec = np.asarray(df[dec_col], dtype=float)

pix = hp.pixelfunc.ang2pix(nside, np.radians(90.0 - dec), np.radians(ra))
npix = hp.nside2npix(nside)
vals = np.zeros(npix)
hist, _ = np.histogram(pix, bins=np.arange(npix + 1))
vals[hist > 0] = hist[hist > 0]
vals[vals < 1] = np.nan

sky_area_deg = 4 * np.pi * (180.0 / np.pi) ** 2
pix_area_arcmin = (sky_area_deg / npix) * 3600
vals = vals / pix_area_arcmin

hp.cartview(
    vals, lonra=[-70, 110], latra=[-70, 10],
    min=1 / pix_area_arcmin, title="",
    cmap="viridis", unit=LBL["density"],
)
plt.show()

## Radius–richness correlation

Cluster radius $R_{\mathrm{amin}}$ as a function of richness $N_{\mathrm{gals}}$, color-coded by photometric redshift $z_{\mathrm{phot}}$. A random subsample of up to 5 000 clusters is shown. The $R_{\mathrm{amin}}$–$N_{\mathrm{gals}}$ scaling encodes the mass–richness relation of the WaZP catalog (Aguena et al. 2021; Benoist et al. 2025).

In [ ]:
query = f"""
SELECT {z_col}, {radius_col}, {ngals_col}
FROM {full_table_name}
WHERE {z_col} IS NOT NULL AND {radius_col} IS NOT NULL AND {ngals_col} IS NOT NULL
ORDER BY RANDOM()
LIMIT 5000
"""
df = pd.DataFrame(db.fetchall_dict(query))

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))
order = np.argsort(df[z_col].astype(float))
ng = df[ngals_col].astype(float).iloc[order].values
radius = df[radius_col].astype(float).iloc[order].values
z = df[z_col].astype(float).iloc[order].values

sc = ax.scatter(ng, radius, c=z, s=10, cmap="plasma", alpha=0.7, linewidths=0)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(LBL["ngals"])
ax.set_ylabel(LBL["radius_mpc"])
cbar = plt.colorbar(sc, ax=ax, label=LBL["zphot"])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## SZE cross-match

Redshift distribution split by external SZE catalog matches (ACT and SPT). Clusters with at least one SZE counterpart are shown in the right panel; optical-only detections are shown in the left panel. Cross-matching with SZE surveys provides an independent confirmation of cluster candidates (Benoist et al. 2025).

$$\text{only optical: } \texttt{sze\_act\_id} = \varnothing \land \texttt{sze\_spt\_id} = \varnothing, \qquad \text{SZE matched: } \texttt{sze\_act\_id} \neq \varnothing \lor \texttt{sze\_spt\_id} \neq \varnothing$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5.0))

conditions = [
    (f"({sze_act_col} IS NULL AND {sze_spt_col} IS NULL)", "Optical only"),
    (f"({sze_act_col} IS NOT NULL OR {sze_spt_col} IS NOT NULL)", "SZE matched"),
]
colors = ["steelblue", "darkorange"]

for ax, (condition, title), color in zip(axes, conditions, colors):
    query = f"""
    SELECT width_bucket({z_col}, 0, 1.2, 24) AS bin, COUNT(*) AS n
    FROM {full_table_name}
    WHERE {z_col} IS NOT NULL AND {condition}
    GROUP BY bin ORDER BY bin
    """
    df = pd.DataFrame(db.fetchall_dict(query))
    bin_edges = np.arange(0, 1.25, 0.05)
    counts = np.zeros(len(bin_edges) - 1)
    for _, row in df.iterrows():
        idx = int(row["bin"]) - 1
        if 0 <= idx < len(counts):
            counts[idx] = row["n"]
    n_total = counts.sum()
    ax.step(bin_edges[:-1], counts, where="post", lw=2, color=color)
    ax.set_title(f"{title}  ($N = {int(n_total):,}$)")
    ax.set_xlabel(LBL["zphot"])
    ax.set_ylabel(LBL["counts"])
    ax.set_xlim(0, 1.2)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Signal-to-noise distribution

Distribution of the WaZP detection signal-to-noise ratio $\mathrm{SNR}$ for the full cluster sample. The $\mathrm{SNR}$ quantifies the overdensity significance of each detection relative to the local background (Aguena et al. 2021). The typical detection threshold for the WaZP catalog is $\mathrm{SNR} \geq 3$ (Benoist et al. 2025).

In [ ]:
query = f"""
SELECT {snr_col}, {ngals_col}
FROM {full_table_name}
WHERE {snr_col} IS NOT NULL
"""
df = pd.DataFrame(db.fetchall_dict(query))
snr_vals = df[snr_col].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, 5.0))

# Linear-scale histogram
axes[0].hist(snr_vals, bins=60, color="steelblue", alpha=0.85, edgecolor="white", linewidth=0.3)
axes[0].axvline(3, color="darkorange", ls="--", lw=1.5, label=r"$\mathrm{SNR} = 3$")
axes[0].axvline(4, color="darkorange", ls=":", lw=1.5, label=r"$\mathrm{SNR} = 4$")
axes[0].axvline(5, color="darkorange", ls="-.", lw=1.5, label=r"$\mathrm{SNR} = 5$")
axes[0].set_xlabel(LBL["snr"])
axes[0].set_ylabel(LBL["counts"])
axes[0].set_title("Linear scale")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Log-scale histogram (cumulative-like for the tail)
axes[1].hist(snr_vals, bins=np.logspace(np.log10(snr_vals.min()), np.log10(snr_vals.max()), 60),
             color="steelblue", alpha=0.85, edgecolor="white", linewidth=0.3)
axes[1].axvline(3, color="darkorange", ls="--", lw=1.5)
axes[1].axvline(4, color="darkorange", ls=":", lw=1.5)
axes[1].axvline(5, color="darkorange", ls="-.", lw=1.5)
axes[1].set_xscale("log")
axes[1].set_xlabel(LBL["snr"])
axes[1].set_ylabel(LBL["counts"])
axes[1].set_title("Log scale")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Cumulative number counts

Cumulative cluster counts $N(>z)$ as a function of photometric redshift for different richness thresholds. The cumulative distribution is a standard test for survey completeness and provides a model-independent view of the cluster abundance (Benoist et al. 2025).

$$N(>z) = \sum_{z_i \geq z} 1, \quad N_{\mathrm{gals}} \geq N_{\mathrm{th}}$$

In [ ]:
ngals_th = (20, 25, 40)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ngals_th)))

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))

for n, color in zip(ngals_th, colors):
    query = f"""
    SELECT {z_col}
    FROM {full_table_name}
    WHERE {ngals_col} >= {n} AND {z_col} IS NOT NULL
    ORDER BY {z_col} DESC
    """
    df = pd.DataFrame(db.fetchall_dict(query))
    if df.empty:
        continue
    z_vals = np.sort(df[z_col].astype(float))
    cumulative = np.arange(len(z_vals), 0, -1)
    ax.step(z_vals, cumulative, where="post", lw=2, color=color,
            label=rf"$N_{{\mathrm{{gals}}}} \geq {n}$  ($N_{{\mathrm{{tot}}}} = {len(z_vals):,}$)")

ax.set_xlabel(LBL["zphot"])
ax.set_ylabel(LBL["cumulative"])
ax.set_xlim(0, 1.2)
ax.set_yscale("log")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Richness–SNR correlation

Cluster richness $N_{\mathrm{gals}}$ versus signal-to-noise ratio $\mathrm{SNR}$, color-coded by photometric redshift. The $N_{\mathrm{gals}}$–$\mathrm{SNR}$ plane reveals the detection significance as a function of cluster richness and helps identify outliers that may warrant further inspection (Aguena et al. 2021; Benoist et al. 2025). A random subsample of up to 5 000 clusters is shown.

In [ ]:
query = f"""
SELECT {z_col}, {ngals_col}, {snr_col}
FROM {full_table_name}
WHERE {z_col} IS NOT NULL AND {ngals_col} IS NOT NULL AND {snr_col} IS NOT NULL
ORDER BY RANDOM()
LIMIT 5000
"""
df = pd.DataFrame(db.fetchall_dict(query))
z = df[z_col].astype(float)
ng = df[ngals_col].astype(float)
snr = df[snr_col].astype(float)

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5.5))
sc = ax.scatter(ng, snr, c=z, s=10, cmap="plasma", alpha=0.7, linewidths=0, vmin=0, vmax=1.2)
ax.set_xscale("log")
ax.set_xlabel(LBL["ngals"])
ax.set_ylabel(LBL["snr"])
cbar = plt.colorbar(sc, ax=ax, label=LBL["zphot"])
ax.axhline(3, color="gray", ls="--", lw=1, alpha=0.5)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()